# Hands-On LangChain #8: Prompt Templates

Companion to [the full article](https://todatabeyond.com/blog/hands-on-langchain-for-llm-applications-development-prompt-templates).

The default path runs real ChatPromptTemplate formatting, message typing, variable validation, literal-brace handling, template reuse, and exact arithmetic checks without API keys. A clearly labelled mocked HTTP transport verifies current OpenAI SDK request/response wiring without contacting the service. Real FLAN-T5-small inference is shown separately; its small-model output is not certified as a good translation. Model downloads on first use require internet. The full updated original hosted workflow follows in an opt-in appendix and was not live-executed. Historical blog outputs are not asserted as current results.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-core>=1.0,<2" "langchain-openai>=1.0,<2" "openai>=2,<3" "transformers>=4.57,<5" torch sentencepiece httpx python-dotenv

## 2. Original customer email and reusable template

In [ ]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

In [ ]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)

In [ ]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [ ]:
from langchain_core.messages import HumanMessage

customer_messages = prompt_template.format_messages(style=customer_style, text=customer_email)
assert prompt_template.input_variables == ["style", "text"]
assert isinstance(customer_messages, list) and isinstance(customer_messages[0], HumanMessage)
assert customer_style in customer_messages[0].content and customer_email in customer_messages[0].content
value = prompt_template.invoke({"style": customer_style, "text": customer_email})
assert value.to_messages() == customer_messages
print("Input variables:", prompt_template.input_variables)
print("Message types:", type(customer_messages).__name__, type(customer_messages[0]).__name__)
print("Customer prompt preserves both supplied values: True")

## 3. Reuse the same template for the original service reply
The source reply blames the customer without supporting evidence. It is preserved as a style-transfer example, not recommended customer-service policy. Template formatting does not repair unfair claims or guarantee a polite response.

In [ ]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [ ]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [ ]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

In [ ]:
assert service_style_pirate in service_messages[0].content
assert service_reply in service_messages[0].content
assert customer_email not in service_messages[0].content
assert prompt_template.input_variables == ["style", "text"]
print("Template reuse: customer and service values remain isolated")

## 4. Missing variables and literal braces
Input data containing braces is not recursively formatted. Literal braces inside the template itself need doubling. Triple backticks delimit data for readability, not as a prompt-injection security boundary.

In [ ]:
try:
    prompt_template.invoke({"style": customer_style})
except KeyError:
    print("Missing text variable: KeyError verified")
else:
    raise AssertionError("Missing variables must fail")

braces = ChatPromptTemplate.from_template('Return JSON like {{"answer": "..."}} for {text}')
message = braces.format_messages(text="Customer wrote {urgent}.")[0]
assert '{"answer": "..."}' in message.content and '{urgent}' in message.content
print("Literal template braces and braces in input data: preserved")

## 5. Original long prompt and deterministic arithmetic check
Prompt reuse is useful, but the supplied student solution is incorrect. This exact arithmetic check does not delegate correctness to an LLM and is not real financial advice.

In [ ]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""

In [ ]:
from decimal import Decimal
correct_rate = Decimal("100") + Decimal("250") + Decimal("10")
student_rate = Decimal("450")
fixed_cost = Decimal("100000")
assert correct_rate == Decimal("360") and student_rate - correct_rate == Decimal("90")
assert correct_rate * Decimal("1000") + fixed_cost == Decimal("460000")
print("Correct first-year formula: 360x + 100000")
print("Student variable-rate overstatement: 90x")
print("For x = 1000: correct cost = 460000")

## 6. OpenAI SDK wiring with an explicit mocked transport

The SDK makes a normal request through httpx.MockTransport, which captures the payload and returns a scripted response. NO OpenAI inference or network call occurs. This checks the maintained helper and attribute-based response access, not translation quality or account/model availability.

In [ ]:
import json
import httpx
from openai import OpenAI

captured_requests = []
def mock_handler(request):
    captured_requests.append(json.loads(request.content))
    return httpx.Response(200, json={
        "id": "chatcmpl-local-fixture", "object": "chat.completion", "created": 0,
        "model": "local-fixture-model", "choices": [{"index": 0, "finish_reason": "stop", "message": {"role": "assistant", "content": "Scripted SDK transport response; not inference."}}],
    })

client = OpenAI(api_key="local-test-only", http_client=httpx.Client(transport=httpx.MockTransport(mock_handler)))
llm_model = "local-fixture-model"
model_options = {}

In [ ]:
def get_completion(prompt, model=llm_model):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        **model_options,
    )
    content = response.choices[0].message.content
    if content is None:
        raise RuntimeError("The model did not return text content.")
    return content

In [ ]:
mock_result = get_completion("Test prompt")
assert captured_requests[0]["messages"] == [{"role": "user", "content": "Test prompt"}]
assert captured_requests[0]["model"] == llm_model
assert "temperature" not in captured_requests[0]
assert mock_result == "Scripted SDK transport response; not inference."
client.close()
print("OpenAI SDK mocked transport: request payload and response text verified (no hosted inference)")

## 7. Real local inference through a prompt/model composition
FLAN-T5-small is a CPU teaching baseline, not an OpenAI chat model. It consumes the formatted prompt as text. Outputs can fail the requested style or omit details; inspect them rather than asserting they are correct translations.

In [ ]:
from pathlib import Path
import tempfile
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM

torch.set_num_threads(2)
cache = str(Path(tempfile.gettempdir()) / "tdb-flan-t5-cache")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", cache_dir=cache)
answer_model.eval()
calls = []

class LocalAnswerLLM(LLM):
    @property
    def _llm_type(self): return "local-flan-t5-small"
    def get_num_tokens(self, text): return len(tokenizer.encode(text))
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=False)
        if inputs["input_ids"].shape[1] > 512:
            raise ValueError("Prompt exceeds this demo's 512-token input budget.")
        with torch.inference_mode():
            tokens = answer_model.generate(**inputs, max_new_tokens=64, do_sample=False)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True)
        calls.append({"prompt": prompt, "answer": answer})
        return answer

llm = LocalAnswerLLM()
print("Model: google/flan-t5-small (CPU, no API key)")

In [ ]:
calls.clear()
customer_chain = prompt_template | llm
local_customer = customer_chain.invoke({"style": customer_style, "text": customer_email})
local_service = customer_chain.invoke({"style": service_style_pirate, "text": service_reply})
assert len(calls) == 2
assert customer_email in calls[0]["prompt"] and customer_style in calls[0]["prompt"]
assert service_reply in calls[1]["prompt"] and service_style_pirate in calls[1]["prompt"]
assert local_customer == calls[0]["answer"] and local_service == calls[1]["answer"]
print("Local customer rewrite:", repr(local_customer))
print("Local service rewrite:", repr(local_service))
print("Real local inference calls:", len(calls))
print("All prompt-template integration assertions passed.")

## 8. Complete optional hosted original workflow

All nineteen original steps follow, with the missing customer-message formatting restored and SDK/import/invoke calls updated. Set OPENAI_API_KEY and OPENAI_CHAT_MODEL locally, then enable the flag. Calls incur charges. Only set OPENAI_TEMPERATURE if your chosen model supports it; zero does not guarantee deterministic or factual output. The historical GPT-3.5 name is not hardcoded as a current default. No hosted calls were executed. Do not commit keys or use private customer emails without permission.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from openai import OpenAI
    from dotenv import load_dotenv
    load_dotenv()
    for name in ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"):
        if not os.environ.get(name):
            raise RuntimeError(f"Set {name} locally before running hosted examples.")
    client = OpenAI()
    # Only set temperature when the selected model supports it.
    model_options = {}
    if os.environ.get("OPENAI_TEMPERATURE") is not None:
        model_options["temperature"] = float(os.environ["OPENAI_TEMPERATURE"])

In [ ]:
if RUN_HOSTED_EXAMPLE:
    llm_model = os.environ["OPENAI_CHAT_MODEL"]

In [ ]:
if RUN_HOSTED_EXAMPLE:
    def get_completion(prompt, model=llm_model):
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            **model_options,
        )
        content = response.choices[0].message.content
        if content is None:
            raise RuntimeError("The model did not return text content.")
        return content

In [ ]:
if RUN_HOSTED_EXAMPLE:
    customer_email = """
    Arrr, I be fuming that me blender lid \
    flew off and splattered me kitchen walls \
    with smoothie! And to make matters worse,\
    the warranty don't cover the cost of \
    cleaning up me kitchen. I need yer help \
    right now, matey!
    """

In [ ]:
if RUN_HOSTED_EXAMPLE:
    style = """American English \
    in a calm and respectful tone
    """
    
    prompt = f"""Translate the text \
    that is delimited by triple backticks 
    into a style that is {style}.
    text: ```{customer_email}```
    """
    
    print(prompt)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    response = get_completion(prompt)
    print(response)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_openai import ChatOpenAI
    chat = ChatOpenAI(model=llm_model, **model_options)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    template_string = """Translate the text \
    that is delimited by triple backticks \
    into a style that is {style}. \
    text: ```{text}```
    """

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_core.prompts import ChatPromptTemplate
    
    prompt_template = ChatPromptTemplate.from_template(template_string)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(prompt_template.messages[0].prompt)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(prompt_template.messages[0].prompt.input_variables)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    customer_style = """American English \
    in a calm and respectful tone
    """
    
    customer_email = """
    Arrr, I be fuming that me blender lid \
    flew off and splattered me kitchen walls \
    with smoothie! And to make matters worse, \
    the warranty don't cover the cost of \
    cleaning up me kitchen. I need yer help \
    right now, matey!
    """

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # The archive omitted this formatting step.
    customer_messages = prompt_template.format_messages(
        style=customer_style, text=customer_email,
    )
    print(type(customer_messages))
    print(type(customer_messages[0]))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # Call the LLM to translate to the style of the customer message
    customer_response = chat.invoke(customer_messages)
    
    print(customer_response.content)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    service_reply = """Hey there customer, \
    the warranty does not cover \
    cleaning expenses for your kitchen \
    because it's your fault that \
    you misused your blender \
    by forgetting to put the lid on before \
    starting the blender. \
    Tough luck! See ya!
    """

In [ ]:
if RUN_HOSTED_EXAMPLE:
    service_style_pirate = """\
    a polite tone \
    that speaks in English Pirate\
    """

In [ ]:
if RUN_HOSTED_EXAMPLE:
    service_messages = prompt_template.format_messages(
        style=service_style_pirate,
        text=service_reply)
    
    print(service_messages[0].content)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    service_response = chat.invoke(service_messages)
    print(service_response.content)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    prompt = f"""
    Determine if the student's solution is correct or not.
    
    Question:
    I'm building a solar power installation and I need \
     help working out the financials. 
    - Land costs $100 / square foot
    - I can buy solar panels for $250 / square foot
    - I negotiated a contract for maintenance that will cost \
    me a flat $100k per year, and an additional $10 / square \
    foot
    What is the total cost for the first year of operations 
    as a function of the number of square feet.
    
    Student's Solution:
    Let x be the size of the installation in square feet.
    Costs:
    1. Land cost: 100x
    2. Solar panel cost: 250x
    3. Maintenance cost: 100,000 + 100x
    Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
    """